# Falcon-H1: attention and SSM dependence in a pretrained parallel hybrid (measurement only)

A standalone Colab notebook for the paper's external check. It asks whether the paper's
measurement pattern appears in a **pretrained production hybrid**: during in-context key–value
recall, is the answer carried causally by attention, by the SSM branch, or by both, and how does
that change with the number of pairs?

**No steering happens here.** Nothing is trained. Present the result as a measurement in a
pretrained parallel hybrid, not as post-convergence control.

**Model.** Falcon-H1 runs attention and Mamba-2 in parallel inside every block and adds their
outputs. Forward hooks on each layer's attention and SSM modules make route-level interventions
possible without editing the model.

**Task.** In-context recall in plain text:

```
Memorize the following pairs.
- apple: red
- river: quiet
...
Now recall one of them.
- river:   ->   quiet
```

Keys and values are single-token English words from two disjoint pools; the queried pair's
position is random. Loads are 2–64 pairs.

**Measurements per load.**
- **Accuracy**, restricted to the value pool (chance = 1/pool size); **in-context accuracy**,
  restricted to the values in that prompt (chance = 1/L); and **bag rate**, how often the
  prediction is *some* in-context value.
- **Route resampling**, as in the paper: every layer's attention (or SSM) output is replaced by
  the output another prompt produced in its own clean pass. D is normalised by headroom above
  chance.
- **Donor-answer rate** under resampling: how often the model outputs the *other* prompt's answer,
  which shows whether the swapped branch carries the retrieved value itself.
- **Deletion**: every attention (or SSM) output set to zero, the ablation used by Borobia et al.
- **Deletion by depth**: attention or SSM zeroed in the first, middle or last third of layers, at
  one load.
- **WikiText-2 perplexity** clean, without attention and without SSM. This doubles as a sanity
  check against the published 3.15× perplexity increase when attention is removed from
  Falcon-H1-0.5B-Base; protocols differ, so expect the same order of magnitude, not the same
  number.

**Run.**
1. `PRESET="smoke"` loads Falcon-H1-Tiny-90M-Instruct with a few prompts, to check the pipeline.
2. `PRESET="full"` loads Falcon-H1-0.5B-Base. Results are saved per load on Google Drive, so a
   disconnect only repeats the load in progress.

**Precision.** On a T4 the model runs in float32 (no native bfloat16); on an L4 or A100 in
bfloat16. Without the Mamba CUDA kernels transformers uses a slower pure-PyTorch SSM path, which
is correct for inference. Send back `FALCON_SUMMARY.txt`.


In [ ]:
import hashlib, json, math, os, pathlib, random, subprocess, sys, time
from contextlib import contextmanager

# ============================== CHOOSE HERE ==============================
PRESET    = "smoke"     # "smoke": Falcon-H1-Tiny-90M-Instruct, few prompts | "full": Falcon-H1-0.5B-Base
MODEL_ID  = None        # optional override, e.g. "tiiuae/Falcon-H1-1.5B-Base"
USE_DRIVE = True
DRIVE_DIR = "/content/drive/MyDrive/falcon_h1_measure"
# =========================================================================

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORKDIR = DRIVE_DIR
else:
    WORKDIR = os.environ.get("FALCON_WORKDIR", str(pathlib.Path.cwd() / "falcon_h1_measure"))

from importlib.metadata import version, PackageNotFoundError


def _ver(pkg):
    try:
        return tuple(int("".join(ch for ch in part if ch.isdigit()) or 0) for part in version(pkg).split(".")[:2])
    except PackageNotFoundError:
        return (0, 0)


if _ver("transformers") < (4, 53):          # FalconH1 support
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers", "accelerate"], check=True)
    raise SystemExit("transformers was upgraded. Runtime -> Restart session, then Run all again.")
if _ver("datasets") == (0, 0):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "datasets"], check=True)

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib as mpl
import matplotlib.pyplot as plt
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer


def P(full, smoke):
    return full if PRESET == "full" else smoke


MODEL = MODEL_ID or P("tiiuae/Falcon-H1-0.5B-Base", "tiiuae/Falcon-H1-Tiny-90M-Instruct")
LOADS           = P([2, 4, 8, 16, 32, 64], [2, 4, 8])
N_SEQ           = P(128, 16)        # prompts per load
LAYER_LOAD      = P(16, 4)          # load used for deletion by depth
N_DEPTH_GROUPS  = 3
N_PPL_WINDOWS   = P(32, 4)
PPL_LEN         = 512
PPL_BATCH       = 4
MAX_BATCH       = 16                # halved automatically on out-of-memory
CACHE_BUDGET_GB = 2.0               # GPU memory for cached branch outputs (both routes)
D_FLOOR         = 0.10              # D undefined when accuracy is within this of chance
SEED            = 0
FORMAT_VERSION  = 2                 # 2: adds the donor-answer rate

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IS_CUDA = DEVICE.type == "cuda"
GPU_NAME = torch.cuda.get_device_name(0) if IS_CUDA else "cpu"
DTYPE = (torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float32) if IS_CUDA else torch.float32
BYTES = 2 if DTYPE == torch.bfloat16 else 4

TAG = f"{MODEL.split('/')[-1]}_{PRESET}"
WORK = pathlib.Path(WORKDIR).expanduser()
RESDIR, FIGDIR, TABDIR = (WORK / f"{n}_{TAG}" for n in ("results", "figures", "tables"))
for _d in (RESDIR, FIGDIR, TABDIR):
    _d.mkdir(parents=True, exist_ok=True)

print(f"colab {IN_COLAB} | {GPU_NAME} | dtype {str(DTYPE).split('.')[-1]} | torch {torch.__version__} "
      f"| transformers {transformers.__version__}")
print(f"preset {PRESET} | model {MODEL} | loads {LOADS} | {N_SEQ} prompts per load")
print(f"work dir {WORK}")


In [ ]:
# Results: append-only JSONL, one record per finished unit (a load, the depth analysis, perplexity),
# keyed by a hash of everything that defines it. Rerunning skips finished units.

def keyof(spec):
    return hashlib.md5(json.dumps(spec, sort_keys=True, default=str).encode()).hexdigest()[:16]


class Records:
    def __init__(self, path):
        self.path, self.recs = path, {}
        if path.exists():
            for line in path.read_text().splitlines():
                try:
                    r = json.loads(line); self.recs[r["_key"]] = r
                except (json.JSONDecodeError, KeyError):
                    pass

    def get(self, spec):
        return self.recs.get(keyof(spec))

    def put(self, spec, payload):
        r = {"_key": keyof(spec), "_spec": spec, "_t": time.time(), **payload}
        self.recs[r["_key"]] = r
        with open(self.path, "a") as f:
            f.write(json.dumps(r, default=float) + "\n"); f.flush(); os.fsync(f.fileno())
        return r


REC = Records(RESDIR / "records.jsonl")
BASE_SPEC = {"model": MODEL, "seed": SEED, "format": FORMAT_VERSION, "dtype": str(DTYPE)}
print(f"{len(REC.recs)} finished record(s) on disk")


In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=DTYPE)
model = model.to(device=DEVICE, dtype=DTYPE).eval()
for p in model.parameters():
    p.requires_grad_(False)


def find_layers(m):
    n = m.config.num_hidden_layers
    lists = [(name, mod) for name, mod in m.named_modules()
             if isinstance(mod, torch.nn.ModuleList) and len(mod) == n]
    assert lists, f"no ModuleList of length num_hidden_layers={n}"
    return lists[0]


def find_branch(layer, preferred, patterns):
    for attr in preferred:
        mod = getattr(layer, attr, None)
        if isinstance(mod, torch.nn.Module):
            return attr, mod
    hits = [(n, c) for n, c in layer.named_children() if any(p in type(c).__name__.lower() for p in patterns)]
    assert len(hits) == 1, (f"cannot identify branch {patterns}; children: "
                            f"{[(n, type(c).__name__) for n, c in layer.named_children()]}")
    return hits[0]


LAYER_PATH, LAYERS = find_layers(model)
ATTN = [find_branch(l, ("self_attn",), ("attention",)) for l in LAYERS]
SSM = [find_branch(l, ("mamba",), ("mamba", "mixer", "ssm")) for l in LAYERS]
N_LAYERS, HIDDEN = len(LAYERS), model.config.hidden_size
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.1f}M parameters | {N_LAYERS} layers at "
      f"'{LAYER_PATH}' | hidden {HIDDEN}")
print(f"attention branch: .{ATTN[0][0]} ({type(ATTN[0][1]).__name__}) | "
      f"SSM branch: .{SSM[0][0]} ({type(SSM[0][1]).__name__})")
print("layer 0 children:", [(n, type(c).__name__) for n, c in LAYERS[0].named_children()])

# ---- route interventions through forward hooks -------------------------------------------------
# spec per route: {"kind": "record", "store": list} | {"kind": "zero"} |
#                 {"kind": "cache", "store": per-layer clean outputs, "perm": batch permutation}
# optional "layers": set of layer indices the intervention applies to.
ACTIVE = {}


def _hook(route, idx):
    def fn(module, inputs, output):
        spec = ACTIVE.get(route)
        if spec is None:
            return None
        if spec.get("layers") is not None and idx not in spec["layers"]:
            return None
        if isinstance(output, tuple):
            t = output[0]
        elif torch.is_tensor(output):
            t = output
        else:
            raise TypeError(f"unexpected {route} output type {type(output)}")
        kind = spec["kind"]
        if kind == "record":
            spec["store"].append(t.detach().clone())
            return None
        if kind == "zero":
            new = torch.zeros_like(t)
        elif kind == "cache":
            new = spec["store"][idx][spec["perm"]]
        else:
            raise ValueError(kind)
        return (new,) + tuple(output[1:]) if isinstance(output, tuple) else new
    return fn


for _h in globals().get("HOOK_HANDLES", []):
    _h.remove()
HOOK_HANDLES = ([mod.register_forward_hook(_hook("attn", i)) for i, (_, mod) in enumerate(ATTN)]
                + [mod.register_forward_hook(_hook("ssm", i)) for i, (_, mod) in enumerate(SSM)])


@contextmanager
def intervene(**routes):
    ACTIVE.clear()
    ACTIVE.update({k: v for k, v in routes.items() if v is not None})
    try:
        yield
    finally:
        ACTIVE.clear()


@torch.no_grad()
def last_logits(x):
    for kw in ({"logits_to_keep": 1}, {"num_logits_to_keep": 1}, {}):
        try:
            out = model(input_ids=x, use_cache=False, **kw)
            return out.logits[:, -1, :].float()
        except TypeError:
            continue
    raise RuntimeError("model forward failed")


def sync():
    if IS_CUDA:
        torch.cuda.synchronize()


In [ ]:
KEY_WORDS = """apple river table window garden doctor pencil island forest bottle mirror candle bridge castle
rocket tiger lemon piano jacket helmet market pocket village teacher engine planet carpet button ticket basket
hammer ladder wallet bucket camera coffee cookie dragon farmer finger flower guitar hunter kitten letter monkey
needle orange parrot pillow puzzle rabbit saddle sister spider statue sugar thunder tomato turtle wagon wizard
anchor arrow badge banner barrel beach beard bell belt bench blanket boat bone book boot bread brick brush cabin
cable cake canal cannon card cart chain chair cheese cherry chest chicken church cloud coat coin comet cotton
crown cup curtain desk diamond dinner dog door drum eagle egg elbow fence field flag flute fork fox frog glass
glove goat grape hat honey horse house iron jar jewel kettle key kite knife lake lamp leaf lion lock map mask
milk moon mouse nail nest net oven owl paint palace paper pearl pepper pig pipe plate pond pot queen rain ring
road robot roof rope rose salt scarf ship shirt shoe shovel sink skirt snake soap sock spoon star stone storm
sword tent tower train tree truck tunnel violin wall whale wheel wolf yard""".split()

VALUE_WORDS = """red blue green yellow purple brown black white pink gray golden silent happy angry brave calm
clever cold dark eager early empty fair fancy fast fierce gentle giant grand great heavy hollow humble icy jolly
kind large late lazy little lonely loud lucky mild modern narrow neat noble odd old plain polite proud quick
quiet rapid rare rich rough round royal rusty sacred safe sharp shiny short shy simple sleepy slow small smooth
soft solid sour spicy steep sticky stormy strange strict strong sunny sweet swift tall tame thick thin tidy tiny
tough ugly vast warm weak wet wide wild wise young bitter bold bright broad busy cheap crisp cruel cute damp deep
dry dusty faint false fresh frozen full funny grumpy hairy harsh hot huge juicy keen lean lively loyal mad merry
messy misty muddy nasty naughty nervous nice pale pure rigid ripe rude salty scary shallow silly skinny spotty
stale stiff sturdy tender tense tricky vivid weird wooden zany""".split()
assert not set(KEY_WORDS) & set(VALUE_WORDS)


def single_token_ids(words, exclude=()):
    ids, kept = [], []
    for w in words:
        e = tok.encode(" " + w, add_special_tokens=False)
        if len(e) == 1 and e[0] not in ids and e[0] not in exclude:
            ids.append(e[0]); kept.append(w)
    return ids, kept


KEY_IDS, KEYS_KEPT = single_token_ids(KEY_WORDS)
VAL_IDS, VALS_KEPT = single_token_ids(VALUE_WORDS, exclude=set(KEY_IDS))
LOADS = [L for L in LOADS if L <= min(len(KEY_IDS), len(VAL_IDS))]
print(f"single-token pools: {len(KEY_IDS)} keys, {len(VAL_IDS)} values | loads used {LOADS}")
CHANCE = 1.0 / len(VAL_IDS)


def piece(s):
    return tok.encode(s, add_special_tokens=False)


BOS = [tok.bos_token_id] if tok.bos_token_id is not None else []
INTRO, RECALL = piece("Memorize the following pairs.\n"), piece("Now recall one of them.\n")
DASH, COLON, NEWLINE = piece("-"), piece(":"), piece("\n")


def make_prompts(load, n, seed):
    # All prompts at a load have identical length, so batches need no padding.
    rng = np.random.default_rng([seed, load, FORMAT_VERSION])
    X, Y, CTX = [], [], []
    for _ in range(n):
        ks = rng.choice(len(KEY_IDS), load, replace=False)
        vs = rng.choice(len(VAL_IDS), load, replace=False)
        q = int(rng.integers(load))
        ids = BOS + INTRO
        for k, v in zip(ks, vs):
            ids += DASH + [KEY_IDS[k]] + COLON + [VAL_IDS[v]] + NEWLINE
        ids += RECALL + DASH + [KEY_IDS[ks[q]]] + COLON
        X.append(ids); Y.append(VAL_IDS[vs[q]]); CTX.append([VAL_IDS[v] for v in vs])
    return torch.tensor(X), torch.tensor(Y), torch.tensor(CTX)


_x, _y, _c = make_prompts(4, 3, 123)
print("\nexample prompt (load 4):\n" + tok.decode(_x[0]) + "   ->" + tok.decode([int(_y[0])]))
_match = sum(tok.encode(tok.decode(row[len(BOS):]), add_special_tokens=False) == row[len(BOS):].tolist()
             for row in _x)
print(f"\nconstructed token ids equal natural re-tokenisation in {_match}/{len(_x)} examples "
      f"(informational; a mismatch only means boundaries differ from free text)")


In [ ]:
# Self-test: hooks fire once per layer, identity resampling reproduces the clean pass, and each
# intervention actually changes the output.
_x, _y, _c = make_prompts(4, 4, 7)
_x = _x.to(DEVICE)
with intervene():
    _base = last_logits(_x)
_rec = {"attn": [], "ssm": []}
with intervene(attn={"kind": "record", "store": _rec["attn"]}, ssm={"kind": "record", "store": _rec["ssm"]}):
    _clean = last_logits(_x)
assert len(_rec["attn"]) == N_LAYERS and len(_rec["ssm"]) == N_LAYERS, \
    f"hooks fired {len(_rec['attn'])}/{len(_rec['ssm'])} times, expected {N_LAYERS}"
_scale = max(1.0, float(_clean.abs().max()))
_tol = (1e-4 if DTYPE == torch.float32 else 2e-3) * _scale
assert float((_base - _clean).abs().max()) <= _tol, "recording changed the output"
_id = torch.arange(4, device=DEVICE)
with intervene(attn={"kind": "cache", "store": _rec["attn"], "perm": _id},
               ssm={"kind": "cache", "store": _rec["ssm"], "perm": _id}):
    _same = last_logits(_x)
assert float((_clean - _same).abs().max()) <= _tol, "identity resampling changed the output"
_diff = lambda a: float((_clean - a).abs().max())
with intervene(attn={"kind": "zero"}):
    _na = last_logits(_x)
with intervene(ssm={"kind": "zero"}):
    _ns = last_logits(_x)
with intervene(attn={"kind": "cache", "store": _rec["attn"], "perm": torch.roll(_id, 1)}):
    _ra = last_logits(_x)
assert _diff(_na) > 10 * _tol and _diff(_ns) > 10 * _tol and _diff(_ra) > 10 * _tol, "an intervention had no effect"
_shapes = (tuple(_rec["attn"][0].shape), tuple(_rec["ssm"][0].shape))
assert _shapes[0] == _shapes[1] == (4, _x.shape[1], HIDDEN), f"unexpected branch output shapes {_shapes}"
print(f"self-test passed: {N_LAYERS} attention and {N_LAYERS} SSM hooks, identity resample exact "
      f"(tol {_tol:.1e}); max logit change: no-attn {_diff(_na):.2f}, no-SSM {_diff(_ns):.2f}, "
      f"attn resample {_diff(_ra):.2f}")
del _rec
if IS_CUDA:
    torch.cuda.empty_cache()


In [ ]:
POOL = torch.tensor(VAL_IDS, device=DEVICE)
CONDS = ["clean", "attn_resample", "ssm_resample", "no_attn", "no_ssm"]


def batch_size_for(T, routes=2):
    per_prompt = T * HIDDEN * N_LAYERS * BYTES * routes
    return int(max(1, min(MAX_BATCH, CACHE_BUDGET_GB * 2 ** 30 // per_prompt)))


@torch.no_grad()
def score(lg, y, ctx, y_donor):
    pool_pred = POOL[lg[:, POOL].argmax(-1)]
    ctx_pred = ctx.gather(1, torch.gather(lg, 1, ctx).argmax(-1, keepdim=True)).squeeze(1)
    logp = torch.log_softmax(lg, -1).gather(1, y[:, None]).squeeze(1)
    return {"acc": float((pool_pred == y).sum()), "acc_ctx": float((ctx_pred == y).sum()),
            "bag": float((pool_pred[:, None] == ctx).any(1).sum()),
            "top1": float((lg.argmax(-1) == y).sum()), "logp": float(logp.sum()),
            "donor": float((pool_pred == y_donor).sum())}   # predicts the donor prompt's answer


def slices(n, B):
    # batches of at least 2 prompts, so every prompt has a donor for resampling
    B = max(2, B)
    sl = [(s, min(s + B, n)) for s in range(0, n, B)]
    if len(sl) > 1 and sl[-1][1] - sl[-1][0] == 1:
        sl[-2] = (sl[-2][0], n); sl.pop()
    return sl


def with_oom_retry(fn, B, label):
    while True:
        try:
            return fn(B)
        except torch.cuda.OutOfMemoryError:
            ACTIVE.clear()
            if IS_CUDA:
                torch.cuda.empty_cache()
            if B <= 2:
                raise
            B //= 2
            print(f"    out of memory in {label}; retrying with batch {B}", flush=True)


def dependence(acc, acc_int):
    if acc - CHANCE < D_FLOOR:
        return float("nan")
    return float(np.clip((acc - acc_int) / (acc - CHANCE), 0.0, 1.0))


def measure_load(L, B):
    X, Y, CTX = make_prompts(L, N_SEQ, SEED)
    T = X.shape[1]
    tot = {c: dict.fromkeys(["acc", "acc_ctx", "bag", "top1", "logp", "donor"], 0.0) for c in CONDS}
    sync(); t0 = time.time()
    for i, j in slices(N_SEQ, B):
        x, y, ctx = X[i:j].to(DEVICE), Y[i:j].to(DEVICE), CTX[i:j].to(DEVICE)
        rec = {"attn": [], "ssm": []}
        with intervene(attn={"kind": "record", "store": rec["attn"]}, ssm={"kind": "record", "store": rec["ssm"]}):
            lg = last_logits(x)
        runs = {"clean": lg}
        perm = torch.roll(torch.arange(x.shape[0], device=DEVICE), 1)       # every prompt gets another's
        y_donor = y[perm]
        with intervene(attn={"kind": "cache", "store": rec["attn"], "perm": perm}):
            runs["attn_resample"] = last_logits(x)
        with intervene(ssm={"kind": "cache", "store": rec["ssm"], "perm": perm}):
            runs["ssm_resample"] = last_logits(x)
        del rec
        with intervene(attn={"kind": "zero"}):
            runs["no_attn"] = last_logits(x)
        with intervene(ssm={"kind": "zero"}):
            runs["no_ssm"] = last_logits(x)
        for c, v in runs.items():
            for k, s in score(v, y, ctx, y_donor).items():
                tot[c][k] += s
        if IS_CUDA:
            torch.cuda.empty_cache()
    sync()
    out = {"load": L, "tokens": T, "batch": max(2, B), "n": N_SEQ, "seconds": time.time() - t0,
           "chance_pool": CHANCE, "chance_ctx": 1.0 / L}
    for c in CONDS:
        for k in ("acc", "acc_ctx", "bag", "top1", "logp", "donor"):
            out[f"{c}_{k}"] = tot[c][k] / N_SEQ
    out["D_attn"] = dependence(out["clean_acc"], out["attn_resample_acc"])
    out["D_ssm"] = dependence(out["clean_acc"], out["ssm_resample_acc"])
    return out


print(f"{'load':>4} {'tok':>4} {'acc':>6} {'ctx':>6} {'bag':>6} | {'attnRes':>7} {'ssmRes':>7} "
      f"{'noAttn':>7} {'noSSM':>7} | {'D_attn':>6} {'D_ssm':>6} | sec")
for L in LOADS:
    spec = {**BASE_SPEC, "block": "load", "load": L, "n": N_SEQ, "pools": [len(KEY_IDS), len(VAL_IDS)]}
    r = REC.get(spec)
    if r is None:
        T_L = make_prompts(L, 1, SEED)[0].shape[1]
        r = REC.put(spec, with_oom_retry(lambda B, L=L: measure_load(L, B), batch_size_for(T_L), f"load {L}"))
    print(f"{L:>4} {r['tokens']:>4} {r['clean_acc']:6.3f} {r['clean_acc_ctx']:6.3f} {r['clean_bag']:6.3f} | "
          f"{r['attn_resample_acc']:7.3f} {r['ssm_resample_acc']:7.3f} {r['no_attn_acc']:7.3f} "
          f"{r['no_ssm_acc']:7.3f} | {r['D_attn']:6.3f} {r['D_ssm']:6.3f} | {r['seconds']:.0f}", flush=True)


In [ ]:
# Deletion by depth at one load: zero attention (or SSM) in the first, middle or last third.
DEPTH_L = min(LOADS, key=lambda L: abs(L - LAYER_LOAD))
GROUPS = [set(g.tolist()) for g in np.array_split(np.arange(N_LAYERS), N_DEPTH_GROUPS)]
GROUP_NAMES = ["first", "middle", "last"] if N_DEPTH_GROUPS == 3 else [f"g{i}" for i in range(N_DEPTH_GROUPS)]


def measure_depth(L, B):
    X, Y, CTX = make_prompts(L, N_SEQ, SEED)
    tot = {}
    for i, j in slices(N_SEQ, B):
        x, y, ctx = X[i:j].to(DEVICE), Y[i:j].to(DEVICE), CTX[i:j].to(DEVICE)
        y_donor = y[torch.roll(torch.arange(x.shape[0], device=DEVICE), 1)]
        for route in ("attn", "ssm"):
            for name, layers in zip(GROUP_NAMES, GROUPS):
                with intervene(**{route: {"kind": "zero", "layers": layers}}):
                    s = score(last_logits(x), y, ctx, y_donor)
                for k, v in s.items():
                    tot[f"no_{route}_{name}_{k}"] = tot.get(f"no_{route}_{name}_{k}", 0.0) + v
    return {"load": L, "groups": [sorted(g) for g in GROUPS], **{k: v / N_SEQ for k, v in tot.items()}}


spec = {**BASE_SPEC, "block": "depth", "load": DEPTH_L, "n": N_SEQ, "groups": N_DEPTH_GROUPS,
        "pools": [len(KEY_IDS), len(VAL_IDS)]}
DEPTH = REC.get(spec) or REC.put(spec, with_oom_retry(lambda B: measure_depth(DEPTH_L, B), MAX_BATCH, "depth"))
_clean_depth = REC.get({**BASE_SPEC, "block": "load", "load": DEPTH_L, "n": N_SEQ,
                        "pools": [len(KEY_IDS), len(VAL_IDS)]})["clean_acc"]
print(f"deletion by depth at load {DEPTH_L} (pool accuracy; clean {_clean_depth:.3f}):")
for route in ("attn", "ssm"):
    print(f"  no {route:<4} " + "  ".join(f"{n}: {DEPTH[f'no_{route}_{n}_acc']:.3f}" for n in GROUP_NAMES))


In [ ]:
# WikiText-2 perplexity: clean, without attention, without SSM.
def measure_ppl(batch):
    from datasets import load_dataset
    try:
        ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="validation")
    except Exception:
        ds = load_dataset("wikitext", "wikitext-2-raw-v1", split="validation")
    text = "\n\n".join(t for t in ds["text"] if t.strip())
    ids = tok.encode(text, add_special_tokens=False)
    windows = [ids[i:i + PPL_LEN] for i in range(0, len(ids) - PPL_LEN, PPL_LEN)][:N_PPL_WINDOWS]
    X = torch.tensor(windows)
    out = {"windows": len(windows), "window_len": PPL_LEN, "batch": batch}
    for cond, routes in [("clean", {}), ("no_attn", {"attn": {"kind": "zero"}}), ("no_ssm", {"ssm": {"kind": "zero"}})]:
        nll, n = 0.0, 0
        for i in range(0, len(X), batch):
            x = X[i:i + batch].to(DEVICE)
            with torch.no_grad(), intervene(**routes):
                lg = model(input_ids=x, use_cache=False).logits[:, :-1].float()
            nll += float(F.cross_entropy(lg.reshape(-1, lg.shape[-1]), x[:, 1:].reshape(-1), reduction="sum"))
            n += x[:, 1:].numel()
            del lg
        out[f"{cond}_ppl"] = math.exp(nll / n)
    return out


spec = {**BASE_SPEC, "block": "ppl", "windows": N_PPL_WINDOWS, "len": PPL_LEN}
PPL = REC.get(spec) or REC.put(spec, with_oom_retry(measure_ppl, PPL_BATCH, "perplexity"))
print(f"WikiText-2 perplexity ({PPL['windows']} x {PPL['window_len']} tokens): clean {PPL['clean_ppl']:.2f}, "
      f"no attention {PPL['no_attn_ppl']:.2f} ({PPL['no_attn_ppl'] / PPL['clean_ppl']:.2f}x), "
      f"no SSM {PPL['no_ssm_ppl']:.2f} ({PPL['no_ssm_ppl'] / PPL['clean_ppl']:.2f}x)")


In [ ]:
LOAD_SPECS = {L: {**BASE_SPEC, "block": "load", "load": L, "n": N_SEQ, "pools": [len(KEY_IDS), len(VAL_IDS)]}
              for L in LOADS}
ROWS = [REC.get(s) for s in LOAD_SPECS.values()]
RES = pd.DataFrame([{k: v for k, v in r.items() if not k.startswith("_")} for r in ROWS if r is not None])
RES.to_csv(TABDIR / "falcon_by_load.csv", index=False)


def f3(v):
    return "--" if v is None or (isinstance(v, float) and np.isnan(v)) else f"{v:.3f}"


# ---- LaTeX ------------------------------------------------------------------------------------
tex = [r"% generated by falcon_h1_measurement.ipynb -- requires booktabs",
       r"\begin{table}[t]", r"\centering", r"\small",
       rf"\caption{{\textbf{{Route dependence in a pretrained parallel hybrid}} ({MODEL.split('/')[-1]}, "
       rf"{N_SEQ} prompts per load, no training). In-context key--value recall with single-token words; "
       rf"accuracy is restricted to the {len(VAL_IDS)}-word value pool (chance {CHANCE:.3f}). "
       rf"Resampled: every layer's attention (SSM) output replaced by another prompt's clean output; "
       rf"deleted: set to zero. $D$ uses the normalisation of Eq.~(2).}}",
       r"\label{tab:falcon}", r"\begin{tabular}{rccccccccc}", r"\toprule",
       r" & \multicolumn{3}{c}{clean} & \multicolumn{2}{c}{resampled} & \multicolumn{2}{c}{deleted} & & \\",
       r"\cmidrule(lr){2-4}\cmidrule(lr){5-6}\cmidrule(lr){7-8}",
       r"pairs & acc. & in-context & bag & attn & SSM & attn & SSM & $D_a$ & $D_s$ \\", r"\midrule"]
for _, r in RES.iterrows():
    tex.append(" & ".join([str(int(r["load"])), f3(r["clean_acc"]), f3(r["clean_acc_ctx"]), f3(r["clean_bag"]),
                           f3(r["attn_resample_acc"]), f3(r["ssm_resample_acc"]), f3(r["no_attn_acc"]),
                           f3(r["no_ssm_acc"]), f3(r["D_attn"]), f3(r["D_ssm"])]) + r" \\")
tex += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
(TABDIR / "falcon_table.tex").write_text("\n".join(tex))

# ---- figure -----------------------------------------------------------------------------------
mpl.rcParams.update({"figure.dpi": 130, "savefig.dpi": 300, "savefig.bbox": "tight", "font.size": 8,
                     "axes.labelsize": 8, "axes.titlesize": 8.5, "legend.fontsize": 7,
                     "xtick.labelsize": 7, "ytick.labelsize": 7,
                     "axes.spines.top": False, "axes.spines.right": False})
COLS = {"clean": "#222222", "attn_resample": "#e07a1f", "ssm_resample": "#1b6ca8",
        "no_attn": "#d1495b", "no_ssm": "#3a9d5d"}
LAB = {"clean": "clean", "attn_resample": "attention resampled", "ssm_resample": "SSM resampled",
       "no_attn": "attention deleted", "no_ssm": "SSM deleted"}
fig, ax = plt.subplots(1, 3, figsize=(7.2, 2.3))
for c in CONDS:
    ls = "--" if c.startswith("no_") else "-"
    ax[0].plot(RES["load"], RES[f"{c}_acc"], ls, marker="o", ms=3, color=COLS[c], label=LAB[c])
    ax[1].plot(RES["load"], RES[f"{c}_bag"], ls, marker="o", ms=3, color=COLS[c])
ax[0].plot(RES["load"], 1.0 / RES["load"], ":", color="0.6", lw=0.8, label="1/L (bag of values)")
ax[2].plot(RES["load"], RES["D_attn"], "-o", ms=3, color=COLS["attn_resample"], label="$D_a$ (attention)")
ax[2].plot(RES["load"], RES["D_ssm"], "-o", ms=3, color=COLS["ssm_resample"], label="$D_s$ (SSM)")
for a, t, yl in [(ax[0], "recall accuracy", "accuracy"), (ax[1], "prediction is an in-context value", "rate"),
                 (ax[2], "causal dependence", "D")]:
    a.set_xscale("log", base=2); a.set_xticks(LOADS); a.set_xticklabels(LOADS)
    a.set_ylim(-0.03, 1.03); a.set_title(t); a.set_xlabel("key-value pairs"); a.set_ylabel(yl)
h0, l0 = ax[0].get_legend_handles_labels()
h2, l2 = ax[2].get_legend_handles_labels()
fig.tight_layout()
fig.legend(h0 + h2, l0 + l2, loc="upper center", ncol=4, frameon=False, bbox_to_anchor=(0.5, 0.0))
for ext in ("pdf", "png"):
    fig.savefig(FIGDIR / f"falcon_route_dependence.{ext}")
plt.show()

# ---- compact summary to paste back --------------------------------------------------------------
cols = ["load", "tokens", "clean_acc", "clean_acc_ctx", "clean_bag", "clean_top1", "clean_logp",
        "attn_resample_acc", "ssm_resample_acc", "no_attn_acc", "no_ssm_acc",
        "attn_resample_bag", "attn_resample_donor", "ssm_resample_bag", "ssm_resample_donor",
        "no_attn_bag", "no_ssm_bag", "no_attn_logp", "no_ssm_logp", "D_attn", "D_ssm", "seconds"]
lines = [f"=== FALCON-H1 ROUTE MEASUREMENT | {time.strftime('%Y-%m-%d %H:%M')} ===",
         f"model {MODEL} | {GPU_NAME} | dtype {str(DTYPE).split('.')[-1]} | transformers {transformers.__version__}",
         f"{N_LAYERS} layers, hidden {HIDDEN} | attention .{ATTN[0][0]} ({type(ATTN[0][1]).__name__}) | "
         f"SSM .{SSM[0][0]} ({type(SSM[0][1]).__name__})",
         f"pools: {len(KEY_IDS)} keys, {len(VAL_IDS)} values (pool chance {CHANCE:.4f}) | {N_SEQ} prompts per load",
         "", RES[cols].round(3).to_string(index=False)]
if "DEPTH" in globals():
    lines += ["", f"deletion by depth at load {DEPTH['load']} (pool accuracy):"]
    for route in ("attn", "ssm"):
        lines.append(f"  no {route:<4} " + "  ".join(f"{n}: {DEPTH[f'no_{route}_{n}_acc']:.3f}" for n in GROUP_NAMES))
if "PPL" in globals():
    lines += ["", f"WikiText-2 perplexity ({PPL['windows']} x {PPL['window_len']}): clean {PPL['clean_ppl']:.2f} | "
                  f"no attention {PPL['no_attn_ppl']:.2f} ({PPL['no_attn_ppl'] / PPL['clean_ppl']:.2f}x) | "
                  f"no SSM {PPL['no_ssm_ppl']:.2f} ({PPL['no_ssm_ppl'] / PPL['clean_ppl']:.2f}x)"]
SUMMARY = "\n".join(lines)
(TABDIR / "FALCON_SUMMARY.txt").write_text(SUMMARY)
print(SUMMARY)
print(f"\nwrote {TABDIR / 'falcon_table.tex'}, {TABDIR / 'FALCON_SUMMARY.txt'} and {FIGDIR}")


## Reading the results

**What each pattern means.**
- **Clean accuracy.** It should be high at small loads for the 0.5B base model. The in-context
  accuracy column shows whether errors are binding errors (a wrong in-context value) or complete
  misses.
- **Attention dependence near 1 with small SSM dependence**, and deletion of attention leaving
  accuracy near the bag-of-values level: in-context recall is carried by attention in a
  pretrained parallel hybrid. That matches "retrieval is the default route" in the paper.
- **SSM dependence rising with load, or both dependences high:** the routes are composed rather
  than alternatives, as in the paper's MQAR result. Report it that way.
- **Bag rate staying high when attention is deleted:** the SSM still carries *which* values
  appeared, but not *which one goes with the key*.

**Framing in the paper.**
- Present it as a measurement in a pretrained parallel hybrid: no intervention is trained and
  nothing is steered.
- Cite the prior attention-ablation and self-speculative decoding work on this model family. The
  perplexity row is a sanity check against their reported ratio, not a new result.
- The post-convergence steering experiment in Falcon-H1 belongs to the revision.
